# CNN-Based Image Regression Analysis — Zucchini (128×128)

Same pipeline as `260305_cnn_regression_analysis`, retargeted to **128×128** images, with a hardened import cell to address the kernel crash seen in the original (§1 below explains what was changed and why).

## 1. Import Libraries

**About the kernel crash:** the original notebook crashed on this exact cell. It already had the standard fix (`KMP_DUPLICATE_LIB_OK=TRUE`) for the classic Windows issue where PyTorch's bundled Intel OpenMP runtime (`libiomp5md.dll`) collides with the one NumPy/SciPy/scikit-image load via MKL — but that env var only *suppresses the abort on detecting a duplicate*, it doesn't remove the underlying conflict, so it doesn't always prevent a crash. This version adds three more layers:

1. **`MKL_THREADING_LAYER=GNU`** — forces MKL to use the GNU OpenMP runtime instead of trying to spin up Intel's (which is what's usually already loaded by PyTorch), removing the conflict at the root instead of just suppressing the crash-on-duplicate check.
2. **Thread-count caps** (`OMP_NUM_THREADS`, `MKL_NUM_THREADS`, `OPENBLAS_NUM_THREADS`, `NUMEXPR_NUM_THREADS`) — reduces the chance of a native thread-oversubscription crash when several BLAS-linked libraries share one process.
3. **`torch.set_num_threads(1)`** right after importing torch, and a try/except around the CUDA check, in case that specific call is what's crashing on a machine with a mismatched GPU driver — falls back to CPU cleanly instead of taking the kernel down with it.

**Important:** these env vars only take effect if set *before* the relevant DLLs are first loaded into the process. If you're seeing this crash, do a full **Restart Kernel** (not just re-run the cell) before running this notebook — re-running the same cell in a kernel that already partially imported these libraries will not help.

If the crash still recurs after a full restart, it points to a deeper binary/version incompatibility in the environment (e.g. torch installed via pip alongside a conda-installed numpy/MKL, or vice versa) that env vars can't fully paper over — at that point the reliable fix is a fresh virtual environment with a pinned, mutually-compatible set of package versions, and checking the actual native crash reason in the Jupyter extension's log (the link in the crash message) rather than guessing further from the notebook side.

In [ ]:
import os

# ── Kernel-crash mitigation: MUST run before importing numpy/torch/scipy/etc ──
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"     # tolerate a duplicate OpenMP runtime instead of aborting
os.environ["MKL_THREADING_LAYER"]  = "GNU"      # avoid the duplicate in the first place: MKL uses GNU OpenMP, not Intel's
os.environ["OMP_NUM_THREADS"]      = "4"        # cap thread pools -- reduces oversubscription crashes
os.environ["MKL_NUM_THREADS"]      = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"]  = "4"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
from PIL import Image
from tqdm import tqdm
from skimage import io
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy import ndimage
from scipy.interpolate import griddata
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib parameters
%matplotlib inline
mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10

# Set random seeds for reproducibility
torch.manual_seed(12)
np.random.seed(12)
torch.set_num_threads(4)   # cap PyTorch's own intra-op threads too, same rationale as above

# Check for GPU -- wrapped defensively: a broken/mismatched CUDA driver can crash
# this call on some machines rather than just returning False.
try:
    _cuda_ok = torch.cuda.is_available()
except Exception as e:
    print(f"⚠ torch.cuda.is_available() raised {type(e).__name__}: {e}")
    print("  Falling back to CPU.")
    _cuda_ok = False

device = torch.device('cuda' if _cuda_ok else 'cpu')
print(f"Using device: {device}")
if _cuda_ok:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("Libraries imported successfully!")
print(f"\nVersions -- torch: {torch.__version__} | numpy: {np.__version__} | "
      f"pandas: {pd.__version__}")
import scipy, sklearn, skimage as _skimage
print(f"scipy: {scipy.__version__} | scikit-learn: {sklearn.__version__} | "
      f"scikit-image: {_skimage.__version__}")

## 2. Configure Parameters

In [ ]:
# Input folder
input_folder = r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\cc\CSVS01"

# Classes to analyze (folder numbers)
folder_numbers = list(range(1, 10))  # 1 to 9

# Use color corrected images
use_cc = False

# Image preprocessing
img_size = (128, 128)  # Resize images to this size -- was (64, 64)
interpolate_outside_circle = False  # Interpolate zero values outside circular disk as preprocessing
preprocessed_folder = r"C:\Users\aymen.masmoudi\Desktop\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\preprocessed"  # Folder for preprocessed images

# Training parameters
batch_size = 32   # if you hit an out-of-memory error at 128x128 (4x the pixels of 64x64), halve this first
num_epochs = 1000
learning_rate = 0.001
train_split = 0.8  # 80% train, 20% validation

if not os.path.exists(input_folder):
    print(f"⚠ WARNING: input_folder does not exist:\n  {input_folder}")
    print("  Double-check the path before continuing -- image loading below will find 0 images otherwise.")

print(f"Input folder: {input_folder}")
print(f"Classes (folders): {folder_numbers}")
print(f"Image size: {img_size}")
print(f"Batch size: {batch_size}")
print(f"Number of epochs: {num_epochs}")
print(f"Learning rate: {learning_rate}")
print(f"Train/Val split: {train_split:.0%}/{1-train_split:.0%}")
print(f"Interpolate outside circle: {interpolate_outside_circle}")
if interpolate_outside_circle:
    print(f"Preprocessed folder: {preprocessed_folder}")

## 3. Load Image Paths

In [ ]:
import os
from glob import glob

def find_images(input_folder, folder_numbers):
    """
    Load all TIFF images from folders named Score=01, Score=02, ...
    Returns a list of (image_path, target_value).
    """
    data_list = []

    for score in folder_numbers:
        folder_name = f"Score={score:02d}"   # 01, 02, ..., 08
        folder_path = os.path.join(input_folder, folder_name)

        if not os.path.isdir(folder_path):
            print(f"Warning: {folder_name} not found")
            continue

        image_paths = sorted(
            glob(os.path.join(folder_path, "*.tif")) +
            glob(os.path.join(folder_path, "*.tiff")) +
            glob(os.path.join(folder_path, "*.TIFF"))
        )

        print(f"{folder_name}: {len(image_paths)} images")

        for img_path in image_paths:
            data_list.append((img_path, float(score)))

    return data_list

# Load image paths and targets
folder_numbers = range(1, 9)   # Scores 1 to 8

data_list = find_images(input_folder, folder_numbers)

print(f"\nTotal images: {len(data_list)}")



# Display target distribution
targets = [target for _, target in data_list]
unique, counts = np.unique(targets, return_counts=True)

plt.figure(figsize=(10, 6))
plt.bar(unique, counts, color='steelblue', edgecolor='black', linewidth=1.5)
plt.xlabel('Target Value (Folder Number)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
plt.title('Distribution of Target Values', fontsize=14, fontweight='bold')
plt.xticks(unique)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 3.1. Preprocess Images (Circle Interpolation)

In [ ]:
def interpolate_circle_exterior(image):
    """
    Interpolate zero values outside the circular disk using nearest non-zero values.
    
    Args:
        image: numpy array of shape (H, W, C) with float values [0, 1]
    
    Returns:
        Interpolated image of same shape
    """
    h, w, c = image.shape
    center_y, center_x = h // 2, w // 2
    radius = min(center_y, center_x)
    
    # Create coordinate grids
    y, x = np.ogrid[:h, :w]
    
    # Create circular mask (True inside circle, False outside)
    distances = np.sqrt((x - center_x)**2 + (y - center_y)**2)
    circle_mask = distances <= radius
    
    # Process each channel separately
    result = np.zeros_like(image)
    
    for channel in range(c):
        img_channel = image[:, :, channel]
        
        # Find zero pixels (outside circle)
        zero_mask = img_channel == 0
        
        # If no zeros or all zeros, return as is
        if not zero_mask.any() or zero_mask.all():
            result[:, :, channel] = img_channel
            continue
        
        # Get non-zero pixels (inside circle)
        non_zero_mask = ~zero_mask
        
        # Coordinates of non-zero pixels
        points = np.column_stack(np.where(non_zero_mask))
        values = img_channel[non_zero_mask]
        
        # Coordinates of zero pixels to interpolate
        zero_points = np.column_stack(np.where(zero_mask))
        
        # Interpolate using nearest neighbor
        interpolated = griddata(points, values, zero_points, method='nearest')
        
        # Build result
        result[:, :, channel] = img_channel.copy()
        result[zero_mask, channel] = interpolated
    
    return result


def preprocess_images_with_interpolation(input_folder, output_folder, folder_numbers, use_cc=True):
    """
    Preprocess all images by applying circle interpolation and save to output folder.
    Skips images that already exist in the output folder.
    
    Args:
        input_folder: Source folder with original images
        output_folder: Destination folder for preprocessed images
        folder_numbers: List of folder numbers to process
        use_cc: Whether to use color-corrected images
    
    Returns:
        Tuple of (total_processed, total_skipped)
    """
    total_processed = 0
    total_skipped = 0
    
    # Create output folder structure
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for folder_num in folder_numbers:
        # Create subfolder for this class
        input_subfolder = os.path.join(input_folder, str(folder_num))
        output_subfolder = os.path.join(output_folder, str(folder_num))
        
        if not os.path.exists(input_subfolder):
            print(f"Warning: Folder '{folder_num}' not found")
            continue
        
        if not os.path.exists(output_subfolder):
            os.makedirs(output_subfolder)
        
        # Find all .tif images
        image_paths = glob(os.path.join(input_subfolder, "*.tif"))
        
        # Filter by color correction status
        cc_filter = '+cc+' if use_cc else '+nocc+'
        filtered_paths = [p for p in image_paths if cc_filter in p]
        
        # Count how many need processing
        paths_to_process = []
        for img_path in filtered_paths:
            output_path = os.path.join(output_subfolder, os.path.basename(img_path))
            if not os.path.exists(output_path):
                paths_to_process.append(img_path)
            else:
                total_skipped += 1
        
        if len(paths_to_process) == 0:
            print(f"Folder {folder_num}: All {len(filtered_paths)} images already preprocessed (skipped)")
        else:
            print(f"Folder {folder_num}: Processing {len(paths_to_process)} images (skipping {len(filtered_paths) - len(paths_to_process)} existing)...")
            
            # Process each image that needs processing
            for img_path in tqdm(paths_to_process, desc=f"Folder {folder_num}"):
                try:
                    # Load image
                    img_array = io.imread(img_path)
                    
                    # Convert to float32 and normalize
                    img_array = img_array.astype(np.float32) / 65535.0
                    
                    # Apply interpolation
                    img_interpolated = interpolate_circle_exterior(img_array)
                    
                    # Convert back to uint16
                    img_uint16 = (img_interpolated * 65535.0).astype(np.uint16)
                    
                    # Save to output folder with same filename
                    output_path = os.path.join(output_subfolder, os.path.basename(img_path))
                    io.imsave(output_path, img_uint16, check_contrast=False)
                    
                    total_processed += 1
                    
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
    
    print(f"\n✓ Preprocessing complete!")
    print(f"  Processed: {total_processed} images")
    print(f"  Skipped: {total_skipped} images (already exist)")
    print(f"  Total: {total_processed + total_skipped} images in {output_folder}")
    return total_processed, total_skipped


# Run preprocessing if enabled
if interpolate_outside_circle:
    print("="*80)
    print("PREPROCESSING: Circle Interpolation")
    print("="*80)
    
    # Always run preprocessing (will skip existing files)
    processed, skipped = preprocess_images_with_interpolation(input_folder, preprocessed_folder, folder_numbers, use_cc)
    
    # Update input_folder to use preprocessed images
    input_folder = preprocessed_folder
    print(f"\n✓ Using preprocessed images from: {input_folder}")
    data_list = find_and_filter_images(input_folder, folder_numbers, use_cc)
    print("="*80)
else:
    print("Circle interpolation disabled. Using original images.")

## 4. Create Custom Dataset

In [ ]:
class ImageRegressionDataset(Dataset):
    """
    Custom Dataset for image regression with optional data augmentation.
    """
    def __init__(self, data_list, transform=None, target_noise_std=0.0):
        """
        Args:
            data_list: List of (image_path, target) tuples
            transform: Optional transform to be applied on images
            target_noise_std: Standard deviation of Gaussian noise to add to targets (for training only)
        """
        self.data_list = data_list
        self.transform = transform
        self.target_noise_std = target_noise_std
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]

        try:
            # Load image
            img_array = io.imread(img_path)

            # Normalize according to image bit depth
            if img_array.dtype == np.uint16:
                img_array = img_array.astype(np.float32) / 65535.0

            elif img_array.dtype == np.uint8:
                img_array = img_array.astype(np.float32) / 255.0

            else:
                raise ValueError(f"Unsupported dtype {img_array.dtype}")

            # If grayscale, convert to 3 channels
            if img_array.ndim == 2:
                img_array = np.stack([img_array] * 3, axis=-1)

            # If image has an alpha channel, remove it
            if img_array.shape[-1] == 4:
                img_array = img_array[:, :, :3]

            # Convert to tensor (C, H, W)
            image = torch.from_numpy(img_array.transpose(2, 0, 1)).float()

            # Apply transforms
            if self.transform:
                image = self.transform(image)

            # Convert target
            target = torch.tensor([target], dtype=torch.float32)

            # Optional target noise
            if self.target_noise_std > 0:
                noise = torch.randn_like(target) * self.target_noise_std
                target = target + noise

            return image, target

        except Exception as e:
            print(f"Error loading image {img_path}: {e}")

            # Create a black tensor image as fallback
            image = torch.zeros(3, img_size[0], img_size[1], dtype=torch.float32)

            if self.transform:
                image = self.transform(image)

            target = torch.tensor([target], dtype=torch.float32)

            return image, target

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2]))
    ]),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Split into train and validation with stratification
# Extract targets for stratification
targets = [target for _, target in data_list]

# Use stratified split to maintain class distribution
train_list, val_list = train_test_split(
    data_list, 
    test_size=1-train_split, 
    stratify=targets,
    random_state=42
)

print(f"Dataset created:")
print(f"  Total samples: {len(data_list)}")
print(f"  Training samples: {len(train_list)}")
print(f"  Validation samples: {len(val_list)}")

# Display class distribution in train and validation sets
train_targets = [target for _, target in train_list]
val_targets = [target for _, target in val_list]

print(f"\nClass distribution in training set:")
train_unique, train_counts = np.unique(train_targets, return_counts=True)
for cls, count in zip(train_unique, train_counts):
    print(f"  Class {int(cls)}: {count} samples ({count/len(train_list)*100:.1f}%)")

print(f"\nClass distribution in validation set:")
val_unique, val_counts = np.unique(val_targets, return_counts=True)
for cls, count in zip(val_unique, val_counts):
    print(f"  Class {int(cls)}: {count} samples ({count/len(val_list)*100:.1f}%)")

# Create datasets (preprocessed images already have interpolation applied if enabled)
train_dataset = ImageRegressionDataset(train_list, transform=train_transform, target_noise_std=0.15)
val_dataset = ImageRegressionDataset(val_list, transform=val_transform, target_noise_std=0.0)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

print(f"\nData loaders created:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

## 5. Visualize Sample Images

In [ ]:
# Get a batch of training data
images, targets = next(iter(train_loader))

print(f"Batch shape: {images.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Targets: {targets.squeeze().tolist()[:8]}")

# Visualize first 8 images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(min(8, len(images))):
    # Denormalize image for display
    img = images[i].permute(1, 2, 0).numpy()
    img = img * 0.5 + 0.5  # Reverse normalization
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(f'Target: {targets[i].item():.1f}', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Training Images with Target Values', fontsize=16, fontweight='bold')
plt.tight_layout()
img = images[i].permute(1, 2, 0).numpy()

print(img.min(), img.max())

img = img * 0.5 + 0.5

print(img.min(), img.max())
plt.show()

## 6. Define CNN Architecture

In [ ]:
class RegressionCNN(nn.Module):
    """
    Custom 2D CNN for regression.
    """
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super(RegressionCNN, self).__init__()
        
        # Convolutional layers
        self.conv_layers = nn.Sequential(
            # Conv Block 1: 3 -> 16
            nn.Conv2d(input_channels, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth),
            nn.ReLU(),
            nn.Conv2d(depth, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 128x128 -> 64x64
            # nn.Dropout2d(0.1),
            
            # Conv Block 2: 16 -> 32
            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2),
            nn.ReLU(),
            nn.Conv2d(depth * 2, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64x64 -> 32x32
            # nn.Dropout2d(0.1),
            
            # Conv Block 3: 32 -> 64
            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4),
            nn.ReLU(),
            nn.Conv2d(depth * 4, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 32x32 -> 16x16
            # nn.Dropout2d(0.1),
            
            # Conv Block 4: 64 -> 128
            nn.Conv2d(depth *4, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8),
            nn.ReLU(),
            nn.Conv2d(depth * 8, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16 -> 8x8
            # nn.Dropout2d(0.1),
            
            # Conv Block 4: 128 -> 256
            nn.Conv2d(depth *8, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16),
            nn.ReLU(),
            nn.Conv2d(depth * 16, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8 -> 4x4
            # nn.Dropout2d(0.1),
            
            # Conv Block 5: 256 -> 512
            nn.Conv2d(depth * 16, depth * 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4 -> 2x2
            # nn.Dropout2d(0.1),
        )
        
        # Calculate the flattened size after conv layers
        # 6 MaxPool2d(2,2) layers -> spatial dims / 64 in each direction;
        # final channel count = depth*32. This formula is algebraically
        # equivalent to (depth*32) * (img_height//64) * (img_width//64) and
        # generalizes correctly to any img_height/img_width divisible by 64
        # (128 included -- verified: ends at 2x2 spatial x (depth*32) channels).
        self.flattened_size = depth * img_height * img_width // 128
        print(f"Flattened size after conv layers: {self.flattened_size}")
        
        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size//2),
            nn.ReLU(),
            nn.Linear(self.flattened_size//2, self.flattened_size//4),
            nn.ReLU(),
            nn.Linear(self.flattened_size//4, 1),  # Single output for regression
            nn.ELU()
        )
    
    def forward(self, x):
        # Convolutional layers
        x = self.conv_layers(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.fc_layers(x)

        # Scale output from range [-inf, inf] to [0, 10]
        # x = torch.sigmoid(x) * 10.0
        return x

# Create model
model = RegressionCNN(input_channels=3, img_height=img_size[0], img_width=img_size[1], depth=16)
model = model.to(device)

# Print model architecture
print("Model Architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 7. Define Loss Function and Optimizer

In [ ]:
# Loss function for regression (Mean Squared Error)
# criterion = nn.MSELoss()
# criterion = nn.HuberLoss()  # More robust to outliers than MSE
class WeightedHuberLoss(nn.Module):
    def __init__(self, coef=1.0):
        super(WeightedHuberLoss, self).__init__()
        self.coef = coef

    def forward(self, pred, target):
        # Weight increases based on distance to 5
        weights = 1 + self.coef * torch.abs(target - 5) / 4
        huber = nn.functional.huber_loss(pred, target, reduction='none')
        return (weights * huber).mean()
# criterion = WeightedHuberLoss(coef=2.0)
criterion = nn.MSELoss()

# Optimizer (Adam)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.0)

# Learning rate scheduler (reduce on plateau)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

# Early stopping
early_stopping_patience = 30  # Number of epochs with no improvement after which training will be stopped

print(f"Loss function: MSELoss")
print(f"Optimizer: Adam (lr={learning_rate}, weight_decay=0)")
print(f"Scheduler: ReduceLROnPlateau (factor=0.1, patience=5)")
print(f"Early stopping patience: {early_stopping_patience} epochs")

## 8. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train for one epoch.
    """
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    
    for images, targets in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        targets = targets.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_mae = mean_absolute_error(all_targets, all_preds)
    epoch_mse = mean_squared_error(all_targets, all_preds)
    
    return epoch_loss, epoch_mae, epoch_mse

def validate_epoch(model, dataloader, criterion, device):
    """
    Validate for one epoch.
    """
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Validation", leave=False):
            images = images.to(device)
            targets = targets.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            # Statistics
            running_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_mae = mean_absolute_error(all_targets, all_preds)
    epoch_mse = mean_squared_error(all_targets, all_preds)
    epoch_r2 = r2_score(all_targets, all_preds)
    
    return epoch_loss, epoch_mae, epoch_mse, epoch_r2, all_preds, all_targets

# Training history
history = {
    'train_loss': [],
    'train_mae': [],
    'train_mse': [],
    'val_loss': [],
    'val_mae': [],
    'val_mse': [],
    'val_r2': [],
    'learning_rate': []
}

best_val_loss = float('inf')
best_model_path = 'best_regression_cnn.pth'
epochs_without_improvement = 0  # Counter for early stopping

print("Starting training...\n")
print("="*80)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss, train_mae, train_mse = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_mae, val_mse, val_r2, _, _ = validate_epoch(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_mae'].append(train_mae)
    history['train_mse'].append(train_mse)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    history['val_mse'].append(val_mse)
    history['val_r2'].append(val_r2)
    history['learning_rate'].append(current_lr)
    # Print metrics
    print(f"  Train Loss: {train_loss:.4f} | Train MAE: {train_mae:.4f} | Train MSE: {train_mse:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val MAE:   {val_mae:.4f} | Val MSE: {val_mse:.4f} | Val R²: {val_r2:.4f}")
    print(f"  Learning Rate: {current_lr:.6f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best model saved (Val Loss: {val_loss:.4f})")
        epochs_without_improvement = 0  # Reset counter
    else:
        epochs_without_improvement += 1
        print(f"  No improvement for {epochs_without_improvement} epoch(s)")

    # Early stopping: stop if no improvement for early_stopping_patience epochs
    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n  ⚠ Early stopping triggered! No improvement for {early_stopping_patience} consecutive epochs.")
        break
    
    print("-"*80)

print(f"Best validation loss: {best_val_loss:.4f}")

print(f"Best model saved to: {best_model_path}")
print("\nTraining complete!")

## 9. Plot Training History

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Use actual number of epochs completed (not num_epochs)
epochs_range = range(1, len(history['train_loss']) + 1)

# Plot loss (Huber Loss)
axes[0].plot(epochs_range, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss (Huber)', fontsize=12, fontweight='bold')
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot MAE
axes[1].plot(epochs_range, history['train_mae'], 'b-', linewidth=2, label='Train MAE')
axes[1].plot(epochs_range, history['val_mae'], 'r-', linewidth=2, label='Val MAE')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE', fontsize=12, fontweight='bold')
axes[1].set_title('Training and Validation MAE', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot MSE
axes[2].plot(epochs_range, history['train_mse'], 'b-', linewidth=2, label='Train MSE')
axes[2].plot(epochs_range, history['val_mse'], 'r-', linewidth=2, label='Val MSE')
axes[2].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[2].set_ylabel('MSE', fontsize=12, fontweight='bold')
axes[2].set_title('Training and Validation MSE', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Plot R²
axes[3].plot(epochs_range, history['val_r2'], 'g-', linewidth=2, label='Val R²')
axes[3].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[3].set_ylabel('R² Score', fontsize=12, fontweight='bold')
axes[3].set_title('Validation R² Score', fontsize=14, fontweight='bold')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

# Plot Learning Rate
axes[4].plot(epochs_range, history['learning_rate'], 'm-', linewidth=2, label='Learning Rate')
axes[4].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[4].set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
axes[4].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[4].set_yscale('log')  # Log scale for better visualization
axes[4].legend()
axes[4].grid(True, alpha=0.3)

# Hide the 6th subplot (empty) -- this call was previously swallowed into a
# comment on the line above (axes[5].axis('off') never actually ran), so the
# empty subplot showed default ticks/frame instead of being hidden.
axes[5].axis('off')

plt.tight_layout()
plt.show()

## 10. Load Best Model and Evaluate

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path))
print("Best model loaded!\n")

# Evaluate on validation set
val_loss, val_mae, val_mse, val_r2, val_preds, val_targets = validate_epoch(model, val_loader, criterion, device)

val_preds = np.array(val_preds).flatten()
val_targets = np.array(val_targets).flatten()

# Calculate additional metrics
val_mse = mean_squared_error(val_targets, val_preds)
val_rmse = np.sqrt(val_mse)

print("="*80)
print("VALIDATION SET PERFORMANCE")
print("="*80)
print(f"Mean Squared Error (MSE):  {val_mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {val_rmse:.4f}")
print(f"Mean Absolute Error (MAE): {val_mae:.4f}")
print(f"R² Score:                  {val_r2:.4f}")
print("="*80)

## 11. Visualize Predictions

In [ ]:
# Scatter plot: Predicted vs True
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Scatter plot
ax = axes[0]
ax.scatter(val_targets, val_preds, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.plot([val_targets.min(), val_targets.max()], 
        [val_targets.min(), val_targets.max()], 
        'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('True Value', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Value', fontsize=12, fontweight='bold')
ax.set_title(f'Predicted vs True Values (R²={val_r2:.3f})', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Residuals
ax = axes[1]
residuals = val_preds - val_targets
ax.scatter(val_targets, residuals, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('True Value', fontsize=12, fontweight='bold')
ax.set_ylabel('Residual (Predicted - True)', fontsize=12, fontweight='bold')
ax.set_title(f'Residual Plot (MAE={val_mae:.3f})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Distribution of predictions by true class
fig, ax = plt.subplots(figsize=(14, 6))

unique_targets = np.unique(val_targets)
positions = []
data_to_plot = []

for target in unique_targets:
    mask = val_targets == target
    data_to_plot.append(val_preds[mask])
    positions.append(target)

bp = ax.boxplot(data_to_plot, positions=positions, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor='lightblue', edgecolor='black', linewidth=1.5),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='black', linewidth=1.5),
                capprops=dict(color='black', linewidth=1.5))

# Add true values as horizontal lines
for target in unique_targets:
    ax.axhline(y=target, color='green', linestyle='--', alpha=0.5, linewidth=1)

ax.set_xlabel('True Class (Folder Number)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Value', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Predictions by True Class', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(unique_targets)

plt.tight_layout()
plt.show()

## 12. Sample Predictions with Images

In [ ]:
# Get a batch of validation data
model.eval()
images, targets = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    predictions = model(images)

images = images.cpu()
predictions = predictions.cpu().numpy().flatten()
targets = targets.numpy().flatten()

# Plot 8 samples
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i in range(min(8, len(images))):
    # Denormalize image
    img = images[i].permute(1, 2, 0).numpy()
    img = img * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    
    true_val = targets[i]
    pred_val = predictions[i]
    error = abs(pred_val - true_val)
    
    # Color code: green if good, red if bad
    color = 'green' if error < 1.0 else 'red'
    
    axes[i].set_title(f'True: {true_val:.1f} | Pred: {pred_val:.2f}\nError: {error:.2f}',
                     fontsize=11, fontweight='bold', color=color)
    axes[i].axis('off')

plt.suptitle('Sample Predictions from Validation Set', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Distribution of predictions by true class - Violin Plot
fig, ax = plt.subplots(figsize=(14, 6))

unique_targets = np.unique(val_targets)
positions = []
data_to_plot = []

for target in unique_targets:
    mask = val_targets == target
    data_to_plot.append(val_preds[mask])
    positions.append(target)

# Create violin plot
vp = ax.violinplot(data_to_plot, positions=positions, widths=0.7, 
                    showmeans=True, showmedians=True, showextrema=True)

# Customize violin plot appearance
for pc in vp['bodies']:
    pc.set_facecolor('lightcoral')
    pc.set_edgecolor('black')
    pc.set_alpha(0.7)
    pc.set_linewidth(1.5)

# Customize statistical lines
vp['cmeans'].set_color('blue')
vp['cmeans'].set_linewidth(2.5)
vp['cmeans'].set_label('Mean')

vp['cmedians'].set_color('red')
vp['cmedians'].set_linewidth(2.5)

vp['cbars'].set_color('black')
vp['cbars'].set_linewidth(1.5)

vp['cmaxes'].set_color('black')
vp['cmaxes'].set_linewidth(1.5)

vp['cmins'].set_color('black')
vp['cmins'].set_linewidth(1.5)

# Add true values as horizontal lines
for target in unique_targets:
    ax.axhline(y=target, color='green', linestyle='--', alpha=0.5, linewidth=1.5)

ax.set_xlabel('True Class (Folder Number)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Value', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Predictions by True Class (Violin Plot)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(unique_targets)
ax.set_ylim([0, 10])  # Set y-axis limits

# Add custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightcoral', edgecolor='black', alpha=0.7, label='Prediction Distribution'),
    plt.Line2D([0], [0], color='red', linewidth=2.5, label='Median Prediction'),
    plt.Line2D([0], [0], color='blue', linewidth=2.5, label='Mean Prediction'),
    plt.Line2D([0], [0], color='green', linestyle='--', linewidth=1.5, label='True Value')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

## 13. Summary and Export Results

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'true_value': val_targets,
    'predicted_value': val_preds,
    'absolute_error': np.abs(val_preds - val_targets),
    'squared_error': (val_preds - val_targets) ** 2
})

print("\n" + "="*80)
print("SUMMARY STATISTICS BY TRUE CLASS")
print("="*80)

for target in sorted(results_df['true_value'].unique()):
    subset = results_df[results_df['true_value'] == target]
    mean_pred = subset['predicted_value'].mean()
    std_pred = subset['predicted_value'].std()
    mean_ae = subset['absolute_error'].mean()
    
    print(f"\nClass {int(target)}:")
    print(f"  Sample count:    {len(subset)}")
    print(f"  Mean prediction: {mean_pred:.3f} ± {std_pred:.3f}")
    print(f"  Mean abs error:  {mean_ae:.3f}")

print("\n" + "="*80)
print("OVERALL PERFORMANCE")
print("="*80)
print(f"Total validation samples: {len(results_df)}")
print(f"MSE:  {val_mse:.4f}")
print(f"RMSE: {val_rmse:.4f}")
print(f"MAE:  {val_mae:.4f}")
print(f"R²:   {val_r2:.4f}")
print("="*80)

# Save results
output_file = "cnn_regression_results.csv"
results_df.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv("cnn_training_history.csv", index=False)
print(f"Training history saved to: cnn_training_history.csv")

In [ ]:
# Predict all validation images
model.eval()
all_images = []
all_true = []
all_preds = []
with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Predicting on validation set"):
        images = images.to(device)
        outputs = model(images)
        
        all_images.extend(images.permute(0, 2, 3, 1).cpu().numpy())  # Convert to (N, H, W, C) for easier handling
        all_true.extend(targets.cpu().numpy())
        all_preds.extend(outputs.cpu().numpy())

all_images = np.array(all_images)
all_true = np.array(all_true)[:,0]
all_preds = np.array(all_preds)[:,0]

## 14. Draw frieze of predictions

In [ ]:
# Frieze of images arranged according to predictions
# One row per score, maximum 10 images per score

sorted_indices = np.argsort(all_preds)
sorted_images = all_images[sorted_indices]
sorted_preds = all_preds[sorted_indices]

max_images_per_score = 10
n_scores = 10  # Scores 0 to 9

fig, axes = plt.subplots(
    n_scores,
    max_images_per_score + 1,
    figsize=(2 * (max_images_per_score + 1), 2.5 * n_scores)
)

# Ensure axes is always 2D
if n_scores == 1:
    axes = np.expand_dims(axes, axis=0)

for score in range(n_scores):

    # Left column: score label
    axes[score, 0].axis('off')
    axes[score, 0].text(
        0.5,
        0.5,
        f"Score {score}",
        fontsize=18,
        fontweight='bold',
        ha='center',
        va='center'
    )

    # Images belonging to this score
    idx = np.where(np.floor(sorted_preds).astype(int) == score)[0]

    # Show at most max_images_per_score images
    idx = idx[:max_images_per_score]

    for j in range(max_images_per_score):

        ax = axes[score, j + 1]
        ax.axis('off')

        if j < len(idx):
            img = sorted_images[idx[j]]
            pred = sorted_preds[idx[j]]

            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)

            ax.imshow(img)
            ax.set_title(f"{pred:.2f}", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Frieze of images arranged according to predictions
# Sort by predicted value
sorted_indices = np.argsort(all_preds)
sorted_images = all_images[sorted_indices]
sorted_preds = all_preds[sorted_indices]
fig, ax = plt.subplots(1, len(sorted_images)+9, figsize=(len(sorted_images)+9, 1))
class_int = 0
for i, (img, pred) in enumerate(zip(sorted_images, sorted_preds)):
    img = img * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    class_int_tmp = int(np.round(pred))
    if class_int_tmp > class_int:
        # Add an intermediate subplot to indicate class change
        ax[i+class_int].axis('off')
        # Add a large text to indicate class int
        ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30, fontweight='bold', ha='center', va='center')
        while class_int < class_int_tmp:
            class_int += 1
            ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
            ax[i+class_int].imshow(img)
            ax[i+class_int].axis('off')
    else:
        ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
        ax[i+class_int].imshow(img)
        ax[i+class_int].axis('off')
while class_int < 9:
    class_int += 1
    ax[len(sorted_images)+class_int-1].axis('off')
    ax[len(sorted_images)+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30, fontweight='bold', ha='center', va='center')

## 15. Draw frieze of annotations

In [ ]:
# Frieze of images arranged according to predictions
# Sort by predicted value
sorted_indices = np.argsort(all_true)
sorted_images = all_images[sorted_indices]
sorted_true = all_true[sorted_indices]
fig, ax = plt.subplots(1, len(sorted_images)+9, figsize=(len(sorted_images)+9, 1))
class_int = 0
for i, (img, true_val) in enumerate(zip(sorted_images, sorted_true)):
    img = img * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    class_int_tmp = int(np.floor(true_val))
    if class_int_tmp > class_int:
        # Add an intermediate subplot to indicate class change
        ax[i+class_int].axis('off')
        # Add a large text to indicate class int
        ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30, fontweight='bold', ha='center', va='center')
        while class_int < class_int_tmp:
            class_int += 1
            ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
            ax[i+class_int].imshow(img)
            ax[i+class_int].axis('off')
    else:
        ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
        ax[i+class_int].imshow(img)
        ax[i+class_int].axis('off')
while class_int < 9:
    class_int += 1
    ax[len(sorted_images)+class_int-1].axis('off')
    ax[len(sorted_images)+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30, fontweight='bold', ha='center', va='center')